In [ ]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, Subset
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import random
import gc

# ------------------------------------------------------
#  Dataset (SAFE streaming mean/std + safe augmentation)
# ------------------------------------------------------

class SpectrogramDataset(Dataset):
    def __init__(self, folder_path, txt_file, mean=None, std=None, augment=False):
        self.folder_path = folder_path
        self.augment = augment
        self.samples = []

        with open(txt_file, "r") as f:
            for line in f:
                file_path, label = line.strip().split()
                full_path = os.path.join(folder_path, file_path)
                self.samples.append((full_path, int(label)))

        # SAFE mean/std calculation (streaming)
        if mean is None or std is None:
            print("Calculating dataset mean/std (CPU-safe streaming)...")
            total_sum = 0.0
            total_sq_sum = 0.0
            total_count = 0

            for path, _ in tqdm(self.samples):
                arr = np.load(path).astype(np.float32)
                total_sum += arr.sum()
                total_sq_sum += (arr ** 2).sum()
                total_count += arr.size

            self.mean = total_sum / total_count
            var = total_sq_sum / total_count - self.mean ** 2
            self.std = float(np.sqrt(max(var, 1e-12)))
        else:
            self.mean = float(mean)
            self.std = float(std)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        file_path, label = self.samples[idx]
        data = np.load(file_path).astype(np.float32)

        # Normalize using global mean/std
        data = (data - self.mean) / (self.std + 1e-6)

        tensor = torch.from_numpy(data).unsqueeze(0)  # (1, H, W)

        # ---- SAFE Augmentation ----
        if self.augment:
            H = tensor.size(1)
            W = tensor.size(2)

            # Frequency masking
            if random.random() < 0.5 and H >= 12:
                mask_h = min(10, H)
                f0 = random.randint(0, H - mask_h)
                tensor[:, f0:f0 + mask_h, :] = 0

            # Time masking
            if random.random() < 0.5 and W >= 12:
                mask_w = min(10, W)
                t0 = random.randint(0, W - mask_w)
                tensor[:, :, t0:t0 + mask_w] = 0

            # Gaussian noise
            if random.random() < 0.5:
                tensor = tensor + 0.02 * torch.randn_like(tensor)

        return tensor, label


# ------------------------------------------------------
#  Improved CNN Model (CPU safe)
# ------------------------------------------------------

class ImprovedCNN(nn.Module):
    def __init__(self, num_classes=9, input_shape=(1, 64, 64)):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        # Dynamically compute flatten size
        with torch.no_grad():
            x = torch.zeros(1, *input_shape)
            x = self.features(x)
            flatten_size = x.view(1, -1).size(1)

        self.classifier = nn.Sequential(
            nn.Linear(flatten_size, 256),
            nn.ReLU(),
            nn.Dropout(0.35),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x


# ------------------------------------------------------
#  Validation (overall + per-class accuracy 0-8)
# ------------------------------------------------------

def evaluate(model, loader, device, num_classes=9):
    model.eval()
    criterion = nn.CrossEntropyLoss()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    class_correct = np.zeros(num_classes, dtype=np.int64)
    class_total = np.zeros(num_classes, dtype=np.int64)

    confusion = np.zeros((num_classes, num_classes), dtype=np.int64)

    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * inputs.size(0)

            preds = outputs.argmax(dim=1)

            total_correct += (preds == labels).sum().item()
            total_samples += labels.size(0)

            # per-class accuracy
            for c in range(num_classes):
                mask = (labels == c)
                class_total[c] += mask.sum().item()
                class_correct[c] += (preds[mask] == c).sum().item()

            # confusion matrix
            for t, p in zip(labels.cpu().numpy(), preds.cpu().numpy()):
                confusion[t, p] += 1

            del outputs, loss, preds

    avg_loss = total_loss / max(total_samples, 1)
    avg_acc = total_correct / max(total_samples, 1)

    class_acc = []
    for c in range(num_classes):
        if class_total[c] == 0:
            class_acc.append(float("nan"))
        else:
            class_acc.append(class_correct[c] / class_total[c])

    return avg_loss, avg_acc, class_acc, confusion


# ------------------------------------------------------
#  Training Function (CPU ONLY)
# ------------------------------------------------------

def train_model():
    folder_path = "/anvil/projects/x-cis220051/corporate/aerospace-rf/fiot_highway2-main"
    txt_file = os.path.join(folder_path, "train.txt")

    # CPU SAFE SETTINGS
    batch_size = 8
    epochs = 40
    learning_rate = 0.001
    num_classes = 9

    # FORCE CPU
    device = torch.device("cpu")
    print("Using device:", device)

    # reproducibility
    random.seed(42)
    np.random.seed(42)
    torch.manual_seed(42)

    # compute mean/std once
    base_ds = SpectrogramDataset(folder_path, txt_file)
    mean, std = base_ds.mean, base_ds.std
    print(f"Global mean={mean:.6f}, std={std:.6f}")

    # full dataset for split
    full_ds = SpectrogramDataset(folder_path, txt_file, mean=mean, std=std, augment=False)
    labels = [label for _, label in full_ds.samples]

    train_idx, val_idx = train_test_split(
        np.arange(len(full_ds)),
        test_size=0.2,
        stratify=np.array(labels),
        random_state=42
    )

    train_ds = Subset(
        SpectrogramDataset(folder_path, txt_file, mean=mean, std=std, augment=True),
        train_idx
    )
    val_ds = Subset(full_ds, val_idx)

    # SAFE DataLoaders (no workers)
    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=False,
        drop_last=True
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=False
    )

    # model
    sample, _ = full_ds[0]
    model = ImprovedCNN(num_classes=num_classes, input_shape=sample.shape).to(device)

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=2
    )

    best_val_acc = 0.0

    # training loop
    for epoch in range(epochs):
        model.train()

        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]"):
            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad(set_to_none=True)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()

            train_loss += loss.item() * inputs.size(0)

            preds = outputs.argmax(dim=1)
            train_correct += (preds == labels).sum().item()
            train_total += labels.size(0)

            del outputs, loss, preds

        train_loss /= max(train_total, 1)
        train_acc = train_correct / max(train_total, 1)

        # validation
        val_loss, val_acc, class_acc, confusion = evaluate(
            model, val_loader, device, num_classes=num_classes
        )

        scheduler.step(val_loss)

        print("\n" + "=" * 70)
        print(
            f"Epoch {epoch+1}/{epochs} | "
            f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}"
        )

        print("\nPer-class accuracy (0 to 8):")
        for c in range(num_classes):
            if np.isnan(class_acc[c]):
                print(f"  Class {c}: N/A (no samples in val)")
            else:
                print(f"  Class {c}: {class_acc[c]:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), "best_model_cpu.pt")
            print(f"\n✅ Saved new best CPU model (val acc = {best_val_acc:.4f})")

        print("=" * 70 + "\n")

        gc.collect()

    print("Training complete.")
    print(f"Best Val Acc: {best_val_acc:.4f}")

    print("\nFinal confusion matrix (rows=true class, cols=pred class):")
    print(confusion)


if __name__ == "__main__":
    train_model()

Using device: cpu
Calculating dataset mean/std (CPU-safe streaming)...


100%|██████████| 12915/12915 [1:29:39<00:00,  2.40it/s]  


Global mean=-32.101116, std=11.108380


Epoch 1/40 [Train]: 100%|██████████| 1291/1291 [1:32:43<00:00,  4.31s/it] 



Epoch 1/40 | Train Loss: 6.2292 Acc: 0.5493 | Val Loss: 1.1734 Acc: 0.7135

Per-class accuracy (0 to 8):
  Class 0: 0.9635
  Class 1: 0.0000
  Class 2: 0.9651
  Class 3: 0.0000
  Class 4: 0.0000
  Class 5: 0.0000
  Class 6: 0.0000
  Class 7: 0.0000
  Class 8: 0.0000

✅ Saved new best CPU model (val acc = 0.7135)



...

Epoch 14/40 [Train]: 100%|██████████| 1291/1291 [37:46<00:00,  1.76s/it]



Epoch 14/40 | Train Loss: 0.9451 Acc: 0.7959 | Val Loss: 0.6790 Acc: 0.7739

Per-class accuracy (0 to 8):
  Class 0: 0.9306
  Class 1: 0.3222
  Class 2: 0.8688
  Class 3: 0.0000
  Class 4: 0.7353
  Class 5: 0.8772
  Class 6: 0.8636
  Class 7: 0.3548
  Class 8: 0.0000



Epoch 15/40 [Train]: 100%|██████████| 1291/1291 [1:21:47<00:00,  3.80s/it]



Epoch 15/40 | Train Loss: 0.9258 Acc: 0.8042 | Val Loss: 0.6606 Acc: 0.7875

Per-class accuracy (0 to 8):
  Class 0: 0.9452
  Class 1: 0.3156
  Class 2: 0.8872
  Class 3: 0.0000
  Class 4: 0.7647
  Class 5: 0.8596
  Class 6: 0.8333
  Class 7: 0.6129
  Class 8: 0.0000



Epoch 16/40 [Train]:  68%|██████▊   | 878/1291 [51:29<30:30,  4.43s/it]  